In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import statsmodels.api as sm
import statsmodels.tools
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

from sklearn.model_selection import train_test_split, KFold, cross_val_score, GroupShuffleSplit
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV
from sklearn import metrics

pd.set_option('display.max_columns', 100)

RANDOM_STATE = 42  
TEST_SIZE = 0.2

In [ ]:
expectancy_data = pd.read_csv('../Life Expectancy Data.csv')


expectancy_data.columns = [c.strip().replace(' ', '_').replace('/', '_') for c in expectancy_data.columns]

expectancy_data.shape

# Explore Data

In [ ]:
expectancy_data.columns

In [ ]:
expectancy_data['Region'].unique()

In [ ]:
# expectancy_data['Country'].unique()

In [ ]:
# TODO: get an overview. head(), dtypes, describe(), shape.
display(expectancy_data.head())
print("\n--- Data Types ---")
print(expectancy_data.dtypes)
display(expectancy_data.describe())
print(f"\nShape: {expectancy_data.shape}")

# Look specifically at which columns are objects and which are numeric
object_cols = expectancy_data.select_dtypes(include=['object']).columns.tolist()
numeric_cols = expectancy_data.select_dtypes(exclude=['object']).columns.tolist()

print(f"\nObject columns: {object_cols}")
print(f"Numeric columns: {numeric_cols}")

In [ ]:
# TODO: look at the distribution of Life Expectancy and at the strongest correlations with it.
sns.histplot(expectancy_data['Life_expectancy'], kde=True)
plt.title('Distribution of LifeExpectancy')
plt.show()

# Calculate correlations for numeric columns only
numeric_df = expectancy_data.select_dtypes(include=[np.number])
correlations = numeric_df.corr()['Life_expectancy'].sort_values(ascending=False)
print("\nStrongest correlations with LifeExpectancy:\n", correlations.head(10))
print("\nWeakest correlations with LifeExpectancy:\n", correlations.tail(5))

In [ ]:
#sns.pairplot(expectancy_data)
#plt.show()

# Check Nulls

In [ ]:
# TODO: count the nulls in every column and show only the columns that have any.
nulls = expectancy_data.isnull().sum()
print(nulls[nulls > 0].sort_values(ascending=False))

# One Hot Encoding Regions

In [ ]:
df = pd.get_dummies(expectancy_data, columns=['Region'], drop_first=True)

In [ ]:
region_columns = [col for col in df.columns if col.startswith('Region_')]
print(region_columns)

# Split Data

In [ ]:
# Model 1: Least Information (Privacy-Preserving)
least_information_columns = [
    "Life_expectancy",
    "Year",
    "Economy_status_Developed",
    "Economy_status_Developing",
    "Population_mln",
    "GDP_per_capita",
    "Schooling"
]

# Model 2: Elaborate (All Features)
# This includes the base features above, plus all sensitive medical/health records.
elaborate_columns = [
    "Life_expectancy",
    "Economy_status_Developed",
    "Economy_status_Developing",
    "Population_mln",
    "GDP_per_capita",
    "Schooling",
    "Infant_deaths",
    "Under_five_deaths",
    "Adult_mortality",
    "Alcohol_consumption",
    "Hepatitis_B",
    "Measles",
    "BMI",
    "Polio",
    "Diphtheria",
    "Incidents_HIV",
    "Thinness_ten_nineteen_years",
    "Thinness_five_nine_years",
    "Year"
] + region_columns

In [ ]:
#least_information_columns = least_base + region_columns
# elaborate_columns = elaborate_base + region_columns

In [ ]:
df_least = df[least_information_columns]
df_elaborate = df[elaborate_columns]

# Country is never a feature, but we need it to group the split (see below)
groups = df['Country']


In [ ]:
print('Region' in df.columns)

In [ ]:
# Removing duplicate info otherwise, model only needs 1 column to infer status

df_elaborate.drop(columns="Economy_status_Developed", inplace=True)

In [ ]:
df_elaborate['GDP_per_capita_log'] = np.log(df_elaborate['GDP_per_capita'])
df_elaborate = df_elaborate.drop(columns=['GDP_per_capita'])

# Both transforms are applied in place, keeping the original column names.
# The app builds its input prompts from the feature list and only knows how
# to map GDP_per_capita_log back to a raw input, so a renamed column would
# make it ask the user to type an already-transformed value.

# Infant_deaths: right-skewed count (skew 1.10, -0.23 once logged).
df_elaborate['Infant_deaths'] = np.log(df_elaborate['Infant_deaths'])

# Adult_mortality: sqrt, not log. This one is about linearity rather than
# skew - correlation with Life_expectancy goes -0.945 raw, -0.957 sqrt,
# -0.954 cbrt, -0.939 log, so log over-corrects and bends it past linear.
df_elaborate['Adult_mortality'] = np.sqrt(df_elaborate['Adult_mortality'])

# Splitting data

In [ ]:
# old one - previous random split, kept for reference only.
# Left commented out so a full run can't overwrite the grouped split below.

# X_train, X_test, y_train, y_test = train_test_split(
#     X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
# )

In [ ]:
TARGET = 'Life_expectancy'

X = df_elaborate.drop(columns=[TARGET])
y = df_elaborate[TARGET]

# Each country contributes 16 yearly rows that barely differ from one another, so a
# random split leaves near-duplicates of every test row sitting in the training set.
# Grouping on Country keeps all of a country's rows on the same side of the split.
#splitter = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_STATE)
#train_idx, test_idx = next(splitter.split(X, y, groups=groups))



#X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
#y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

X_train, X_test, y_train, y_test = train_test_split(
     X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
 )

print('X_train:', X_train.shape)
print('X_test: ', X_test.shape)
#print('Countries in both splits:', len(set(groups.iloc[train_idx]) & set(groups.iloc[test_idx])))
print('Features:', list(X_train.columns))

# Feature Engineering

In [ ]:
def add_gdp_log(X_df):
    X_df = X_df.copy()
    if 'GDP_per_capita' in X_df.columns:
        X_df['GDP_per_capita_log'] = np.log(X_df['GDP_per_capita'])
        X_df = X_df.drop(columns=['GDP_per_capita'])
    return X_df
X_train = add_gdp_log(X_train)
X_test  = add_gdp_log(X_test)

X_train.head()

# Fit Model and Evaluate Baseline RMSE

In [ ]:
def fit_and_score(X_tr, X_te, y_tr, y_te, label=''):
    m = LinearRegression()
    m.fit(X_tr, y_tr)
    preds = m.predict(X_te)
    rmse = metrics.root_mean_squared_error(y_te, preds)
    r2 = metrics.r2_score(y_te, preds)
    if label:
        print(f'{label:<40} RMSE: {rmse:.4f}   R2: {r2:.4f}')
    return m, rmse


results = {}

baseline_model, baseline_rmse = fit_and_score(
    X_train, X_test, y_train, y_test, 'Baseline (all features)'
)
results['Baseline (all features)'] = baseline_rmse



# Validate Columns to Drop

In [ ]:
# stepwise

def stepwise_selection(X, y, threshold_in = 0.01, threshold_out = 0.05, verbose = True):
    # The function is checking for p-values (whether features are statistically significant) - lower is better
    included = [] # this is going to be the list of features we keep
    while True:
        changed = False
        # forward step
        excluded = list(set(X.columns) - set(included))
        new_pval = pd.Series(index = excluded, dtype = 'float64')
        for new_column in excluded:
            model = sm.OLS(y, sm.add_constant(pd.DataFrame(X[included + [new_column]]))).fit()
            new_pval[new_column] = model.pvalues[new_column]
        best_pval = new_pval.min()
        # we add the feature with the lowest (best) p-value under the threshold to our 'included' list
        if best_pval < threshold_in:
            best_feature = new_pval.idxmin()
            included.append(best_feature)
            changed = True
            if verbose:
                print('Add  {:30} with p-value {:.6}'.format(best_feature, best_pval)) # specifying the verbose text


        # backward step: removing features if new features added to the list make them statistically insignificant
        model = sm.OLS(y, sm.add_constant(pd.DataFrame(X[included]))).fit()

        # use all coefs except intercept
        pvalues = model.pvalues.iloc[1:]
        worst_pval = pvalues.max() # null if pvalues is empty
        # if the p-value exceeds the upper threshold, the feature will be dropped from the 'included' list
        if worst_pval > threshold_out:
            changed = True
            worst_feature = pvalues.idxmax()
            included.remove(worst_feature)
            if verbose:
                print('Drop {:30} with p-value {:.6}'.format(worst_feature, worst_pval))
        if not changed:
            break
    return included

In [ ]:
X_train_f = X_train.astype(float)  

sel = stepwise_selection(X_train_f, y_train)

print('\nSelected', len(sel), 'of', X_train.shape[1], ':')
print(sel)
print('\nExcluded:', sorted(set(X_train.columns) - set(sel)))

In [ ]:
def score(cols, label):
    m = LinearRegression().fit(X_train[cols], y_train)
    tr = metrics.root_mean_squared_error(y_train, m.predict(X_train[cols]))
    te = metrics.root_mean_squared_error(y_test, m.predict(X_test[cols]))
    print(f'{label:<28} train {tr:.4f} | test {te:.4f}')
    return m, te

score(list(X_train.columns), 'All features')
score(sel, 'Stepwise selected')

# RMSE score for the  columns i initially thought of excluding 
redundant = ['Infant_deaths', 'Diphtheria', 'Thinness_five_nine_years']
score([c for c in X_train.columns if c not in redundant], 'Manual redundancy drops')

In [ ]:
stepwise_model = LinearRegression().fit(X_train[sel], y_train)
preds = stepwise_model.predict(X_test[sel])

rmse = metrics.root_mean_squared_error(y_test, preds)
print('Stepwise RMSE:', round(rmse, 4))
print('R2:', round(metrics.r2_score(y_test, preds), 4))

In [ ]:
def calculate_vif(X, thresh=10.0):
    X = X.astype(float)
    variables = list(X.columns)
    while True:
        Xc = add_constant(X[variables])
        vif = pd.Series(
            [variance_inflation_factor(Xc.values, i) for i in range(Xc.shape[1])],
            index=Xc.columns
        ).drop('const')
        if vif.max() <= thresh:
            break
        worst = vif.idxmax()
        print(f'dropping {worst} VIF={vif.max():.2f}')
        variables.remove(worst)
    print('Remaining:', variables)
    return variables

In [ ]:
# rmse using VIF on the results from stepwise

sel_clean = calculate_vif(X_train[sel])
m = LinearRegression().fit(X_train[sel_clean], y_train)
print('RMSE:', metrics.root_mean_squared_error(y_test, m.predict(X_test[sel_clean])))

In [ ]:
# testing adding back all regions

sel_final = sel_clean + ['Region_Asia', 'Region_Middle East',
                          'Region_North America', 'Region_Rest of Europe']

m = LinearRegression().fit(X_train[sel_final], y_train)
print('RMSE:', metrics.root_mean_squared_error(y_test, m.predict(X_test[sel_final])))

In [ ]:
# final model: stepwise selection (sel) - VIF pruning above was tested and rejected

X_train_sm = sm.add_constant(X_train[sel].astype(float))

ols_model = sm.OLS(y_train, X_train_sm).fit()
ols_model.summary()

# Extended stats

Extends the elaborate notebook with residual diagnostics and a held-out test
report for the final model. Everything above this section is unchanged.

## Residual diagnostics

Residuals vs fitted (linearity and even spread), residual distribution, and
Q-Q plot (normality).

In [ ]:
resid, fitted_vals = ols_model.resid, ols_model.fittedvalues

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].scatter(fitted_vals, resid, s=8, alpha=0.5)
ax[0].axhline(0, color="red", lw=1)
ax[0].set(title="Residuals vs fitted", xlabel="Fitted (years)", ylabel="Residual")
ax[1].hist(resid, bins=40)
ax[1].set(title="Residual distribution", xlabel="Residual (years)")
sm.qqplot(resid, line="45", fit=True, ax=ax[2])
ax[2].set_title("Q-Q plot")
plt.tight_layout()
plt.show()

Patternless scatter around zero, a near bell-shaped distribution, and Q-Q
points close to the line: the assumptions hold well enough for prediction. A
formal spread test does flag mild non-constant variance, which affects the
precision of coefficient p-values, not the predictions themselves.

## Held-out test report

The final model scored once on the 20% of records never used in fitting,
plus the 95% give-or-take range for a single prediction.

In [ ]:
X_test_sm = sm.add_constant(X_test[sel].astype(float))
pred = ols_model.predict(X_test_sm)

pi = ols_model.get_prediction(X_test_sm).summary_frame(alpha=0.05)
give_take = float((pi["obs_ci_upper"] - pi["obs_ci_lower"]).mean()) / 2

ss_res = ((y_test - pred) ** 2).sum()
ss_tot = ((y_test - y_test.mean()) ** 2).sum()
print(f"Test RMSE  {np.sqrt(ss_res / len(y_test)):.3f} years")
print(f"Test MAE   {np.abs(y_test - pred).mean():.3f} years")
print(f"Test R2    {1 - ss_res / ss_tot:.3f}")
print(f"95% give-or-take for one prediction  +/- {give_take:.1f} years")